# CPSC 368 – Group 4: YouTube Data Collection
## Phase 3 – Data Collection via YouTube Data API v3

This notebook collects channel stats, video stats, and top comments for four political YouTube channels:

| Channel | Lean |
|---|---|
| Fox News | Conservative |
| The Young Turks | Liberal |

**Output:** Three CSV files (`channels.csv`, `videos.csv`, `comments.csv`) ready for Oracle ingestion.


## 1. Configuration

In [29]:
import time
import csv
from googleapiclient.discovery import build
import json
import pandas as pd
import os

from dotenv import load_dotenv

In [30]:
load_dotenv()
API_KEY = os.getenv("API_KEY")

In [ ]:
CHANNELS = [
    # {"id": "UCXIJgqnII2ZOINSWNOGFThA", "name": "Fox News",          "lean": "conservative"},
    # {"id": "UCupvZG-5ko_eiXAupbDfxWw", "name": "CNN",               "lean": "liberal"},
    {"id": "UCnQC_G5Xsjhp9fEJKuIcrSw", "name": "Ben Shapiro", "lean": "conservative"},
    {"id": "UC1yBKRuGpC1tSM73A0ZjYjQ", "name": "The Young Turks", "lean": "liberal"},
]
START_YEAR = 2016
END_YEAR = 2025

# output file names
OUT_CHANNELS = "../../cleaned_data/youtube/channels.csv"
OUT_VIDEOS = "../../cleaned_data/youtube/videos.csv"
OUT_COMMENTS = "../../cleaned_data/youtube/comments.csv"

os.makedirs("../../cleaned_data/youtube", exist_ok=True)

# setup youtube api
youtube = build("youtube", "v3", developerKey=API_KEY)

## 3. Fetch Channel Stats

In [32]:
def fetch_channel_stats(channel_id, lean):
    request = youtube.channels().list(
        part="snippet,contentDetails,statistics",
        id=channel_id,
    )

    resp = request.execute()
    if not resp or not resp.get("items"):
        return None
    item = resp["items"][0]
    stats = item.get("statistics", {})
    return {
        "channel_id": channel_id,
        "channel_name": item["snippet"]["title"],
        "lean": lean,
        "description": item["snippet"].get("description", ""),
        "published_at": item["snippet"].get("publishedAt", ""),
        "subscriber_count": stats.get("subscriberCount", ""),
        "total_view_count": stats.get("viewCount", ""),
        "total_video_count": stats.get("videoCount", ""),
        "uploads_playlist": item["contentDetails"]["relatedPlaylists"]["uploads"],
    }


# collect channel stats for all 4 channels
all_channels = []
for ch in CHANNELS:
    print(f"Fetching stats for: {ch['name']}...")
    stats = fetch_channel_stats(ch["id"], ch["lean"])
    if stats:
        all_channels.append(stats)
        print(f"  Subscribers : {stats['subscriber_count']}")
        print(f"  Total views : {stats['total_view_count']}")
        print(f"  Total videos: {stats['total_video_count']}")
    print()

print(f"Channel stats collected for {len(all_channels)} channels.")

Fetching stats for: Ben Shapiro...
  Subscribers : 7090000
  Total views : 4646610341
  Total videos: 9727

Fetching stats for: The Young Turks...
  Subscribers : 6500000
  Total views : 7307837549
  Total videos: 71035

Channel stats collected for 2 channels.


In [33]:
df = pd.json_normalize(all_channels)
df.to_csv(OUT_CHANNELS, index=False)
display(df)

,channel_id,channel_name,lean,description,published_at,subscriber_count,total_view_count,total_video_count,uploads_playlist
0,UCnQC_G5Xsjhp9fEJKuIcrSw,Ben Shapiro,conservative,Ben Shapiro is a renowned conservative politic...,2016-11-01T00:13:29Z,7090000,4646610341,9727,UUnQC_G5Xsjhp9fEJKuIcrSw
1,UC1yBKRuGpC1tSM73A0ZjYjQ,The Young Turks,liberal,The Young Turks is The Online News Show. Join ...,2005-12-21T20:46:51Z,6500000,7307837549,71035,UU1yBKRuGpC1tSM73A0ZjYjQ


## 5. Fetch Video IDs

In [37]:
def fetch_all_videos_for_channel(channel_name, uploads_playlist_id):
    os.makedirs("checkpoints", exist_ok=True)
    checkpoint_file = (
        f"checkpoints/{channel_name.lower().replace(' ', '_').replace('/', '')}.json"
    )

    # checkpoint file incase we stopped running for a channel midway
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, "r") as f:
            checkpoint = json.load(f)
        video_ids = checkpoint["videos"]
        next_page = checkpoint["last_page_token"]
        page_num = checkpoint["last_page_num"]
        already_seen = {v["video_id"] for v in video_ids}
        print(
            f"Resuming {channel_name} from page {page_num} ({len(video_ids)} videos already collected)"
        )
    else:
        video_ids = []
        next_page = None
        page_num = 0
        already_seen = set()
        print(f"Starting fresh for {channel_name}")

    done = False
    while not done:
        resp = (
            youtube.playlistItems()
            .list(
                part="contentDetails,snippet",
                playlistId=uploads_playlist_id,
                maxResults=50,
                pageToken=next_page,
            )
            .execute()
        )

        if not resp:
            print(f"Error on page {page_num} for {channel_name}")
            break

        for item in resp.get("items", []):
            pub = item["snippet"].get("publishedAt", "")
            if not pub:
                continue
            year = int(pub[:4])
            video_id = item["contentDetails"]["videoId"]
            video_title = item["snippet"].get("title", "")

            if year < START_YEAR:
                print(f"Reached pre-{START_YEAR} content — stopping.")
                done = True
                break

            if year > END_YEAR or video_id in already_seen:
                continue

            video_ids.append(
                {
                    "video_id": video_id,
                    "title": video_title,
                    "published_at": pub,
                    "page": page_num,
                }
            )
            already_seen.add(video_id)

        next_page = resp.get("nextPageToken")
        page_num += 1

        # Save checkpoint after every page
        with open(checkpoint_file, "w") as f:
            json.dump(
                {
                    "channel_name": channel_name,
                    "last_page_token": next_page,
                    "last_page_num": page_num,
                    "videos": video_ids,
                },
                f,
            )

        if page_num % 10 == 0:
            print(
                f"Finished page {page_num}, {len(video_ids)} IDs so far. Currently at {pub}"
            )

        if not next_page:
            break

        time.sleep(0.5)

    print(f"Found {len(video_ids)} videos in range for {channel_name}")
    return video_ids

### one channel at a time

In [35]:
all_channels_df = pd.read_csv(OUT_CHANNELS)
all_channels_df

,channel_id,channel_name,lean,description,published_at,subscriber_count,total_view_count,total_video_count,uploads_playlist
0,UCnQC_G5Xsjhp9fEJKuIcrSw,Ben Shapiro,conservative,Ben Shapiro is a renowned conservative politic...,2016-11-01T00:13:29Z,7090000,4646610341,9727,UUnQC_G5Xsjhp9fEJKuIcrSw
1,UC1yBKRuGpC1tSM73A0ZjYjQ,The Young Turks,liberal,The Young Turks is The Online News Show. Join ...,2005-12-21T20:46:51Z,6500000,7307837549,71035,UU1yBKRuGpC1tSM73A0ZjYjQ


In [ ]:
"""
If this is commented out it's because this part is done, data can be found in scripts/youtube/checkpoints/
"""

# channel_name = 'Ben Shapiro'
# channel_id = all_channels_df[all_channels_df["channel_name"] == channel_name]["channel_id"].values[0]
# uploads_playlist_id = all_channels_df[all_channels_df["channel_name"] == channel_name]["uploads_playlist"].values[0]

# ids = fetch_all_videos_for_channel(channel_name, uploads_playlist_id)

# channel_name = 'The Young Turks'
# channel_id = all_channels_df[all_channels_df["channel_name"] == channel_name]["channel_id"].values[0]
# uploads_playlist_id = all_channels_df[all_channels_df["channel_name"] == channel_name]["uploads_playlist"].values[0]

# ids = fetch_all_videos_for_channel(channel_name, uploads_playlist_id)

Starting fresh for Ben Shapiro
Finished page 10, 251 IDs so far. Currently at 2025-10-09T15:31:11Z
Finished page 20, 751 IDs so far. Currently at 2025-06-21T15:00:23Z
Finished page 30, 1251 IDs so far. Currently at 2025-04-02T22:00:50Z
Finished page 40, 1751 IDs so far. Currently at 2025-01-25T00:00:04Z
Finished page 50, 2251 IDs so far. Currently at 2024-11-11T23:30:14Z
Finished page 60, 2751 IDs so far. Currently at 2024-09-05T21:30:03Z
Finished page 70, 3251 IDs so far. Currently at 2024-06-25T19:00:33Z
Finished page 80, 3751 IDs so far. Currently at 2024-03-28T01:30:22Z
Finished page 90, 4251 IDs so far. Currently at 2023-12-14T23:00:35Z
Finished page 100, 4751 IDs so far. Currently at 2023-09-06T17:00:09Z
Finished page 110, 5251 IDs so far. Currently at 2023-05-25T21:30:06Z
Finished page 120, 5751 IDs so far. Currently at 2023-02-08T00:15:00Z
Finished page 130, 6251 IDs so far. Currently at 2022-09-14T01:00:14Z
Finished page 140, 6751 IDs so far. Currently at 2022-04-13T17:00:12Z


## 5.5 Showcasing Video ID DF

In [39]:
with open("checkpoints/the_young_turks.json", "r") as f:
    tyt_data = json.load(f)

with open("checkpoints/ben_shapiro.json", "r") as f:
    bs_data = json.load(f)

print("========= The Young Turks =========")
tyt_df = pd.DataFrame(tyt_data["videos"])
display(pd.concat([tyt_df.head(5), tyt_df.tail(5)]))

print("========= Ben Shapiro =========")
bs_df = pd.DataFrame(bs_data["videos"])
display(pd.concat([bs_df.head(5), bs_df.tail(5)]))

========= The Young Turks =========


,video_id,title,published_at,page
0,b_AFY4cbnYs,"Nvidia, Palantir Take MASSIVE HIT",2025-12-31T05:15:05Z,12
1,WYaJAl8QtoQ,You Won't BELIEVE How Powerful Jeffrey Epstein...,2025-12-31T04:30:19Z,12
2,3r8xkgx5RWA,We're Paying For Israel's Wars AGAIN,2025-12-31T03:45:00Z,12
3,Tt1raC3AhtE,Alan Dershowitz Has Lost His Damn Mind,2025-12-31T03:00:41Z,12
4,YNgcVPmlDIA,EXPOSED: Israel Behind Iran Protests?!,2025-12-31T02:15:09Z,12
13859,n7-uJC92QgY,The Young Turks 11.16.17: Al Franken Allegatio...,2017-11-17T03:43:00Z,289
13860,oLeZXeY53Rk,"The Young Turks 11.14.17: Wikileaks, Sessions,...",2017-11-15T02:38:00Z,289
13861,QJXkt_XiL68,"The Young Turks 11.13.17: Feinstein, Big Pharm...",2017-11-14T02:07:00Z,289
13862,2Qu79o3CWUY,The Young Turks 11.10.17: Roy Moore & Louis CK,2017-11-11T22:52:00Z,289
13863,j25IOnmEp0k,"The Young Turks 11.09.17: Roy Moore, Trump in ...",2017-11-10T01:51:00Z,289


========= Ben Shapiro =========


,video_id,title,published_at,page
0,cXSaG9rvDhY,The WORST Solution To Your Life,2025-12-31T20:18:03Z,4
1,vB4lecMG8hY,Ben Shapiro Reacts To Most Viral Moments of 2025,2025-12-31T17:49:00Z,5
2,I1Cqub0iuxU,The Face Piercings Never Lie,2025-12-30T21:59:48Z,5
3,Fm7Niep35FM,THESE Are The Words Of The Year?,2025-12-30T18:50:43Z,5
4,pruCuydnLhw,The New Nuclear Arms Race,2025-12-30T15:01:41Z,5
9469,g5wEep7nLOg,Ep. 208 - President-Elect Trump,2016-11-11T00:32:42Z,194
9470,gaQ8ISE5E2Y,Ep. 207 - Today Is The Day,2016-11-11T00:32:38Z,194
9471,HMYpHlxChmo,Ep. 206 - The Final Prediction: Trump vs. Clinton,2016-11-11T00:32:32Z,194
9472,g29TugDYLRw,Ep. 205 - The Final Countdown,2016-11-07T05:49:30Z,194
9473,PY09dED8404,Ep. 204 - Will Trump Provide Another Cubs-Like...,2016-11-07T05:49:26Z,194


## 6. Fetch Video Stats (all videos)

In [ ]:
"""
If this is commented out it's because this part is done, data can be found in output/videos.csv
"""

# CHECKPOINT_FILES = {
#     "ben_shapiro":    "checkpoints/ben_shapiro.json",
#     "the_young_turks": "checkpoints/the_young_turks.json",
# }

# CHANNEL_META = {
#     "ben_shapiro":     {"channel_id": "UCnQC_G5Xsjhp9fEJKuIcrSw", "lean": "conservative"},
#     "the_young_turks": {"channel_id": "UC1yBKRuGpC1tSM73A0ZjYjQ", "lean": "liberal"},
# }

# VIDEO_OUT_FILE = "output/videos.csv"

# all_entries = []

# for channel_name, filepath in CHECKPOINT_FILES.items():
#     with open(filepath, "r") as f:
#         checkpoint = json.load(f)
#     videos = checkpoint["videos"]
#     for v in videos:
#         all_entries.append({
#             "video_id":     v["video_id"],
#             "published_at": v["published_at"],
#             "channel_key":  channel_name,
#         })

# print(f"Loading stats for {len(all_entries)} videos")
# # lookup video stats for all videos
# stats_lookup = {}
# ids_only = [e["video_id"] for e in all_entries]
# total_batches = (len(ids_only) + 49) // 50

# print(f"Fetching stats in {total_batches} batches of 50")

# for i in range(0, len(ids_only), 50):
#     batch = ids_only[i:i+50]
#     batch_num = i // 50 + 1

#     resp = youtube.videos().list(
#         part="statistics",
#         id=",".join(batch)
#     ).execute()

#     for item in resp.get("items", []):
#         stats = item.get("statistics", {})
#         stats_lookup[item["id"]] = {
#             "view_count":    int(stats.get("viewCount",    0) or 0),
#             "like_count":    int(stats.get("likeCount",    0) or 0),
#             "comment_count": int(stats.get("commentCount", 0) or 0),
#         }

#     if batch_num % 50 == 0 or batch_num == total_batches:
#         print(f"  Batch {batch_num}/{total_batches} done, {len(stats_lookup)} stats fetched so far")

#     time.sleep(0.3)

# print(f"\nStats fetched for {len(stats_lookup)} videos.")
# print(f"Videos with no stats (deleted/private): {len(all_entries) - len(stats_lookup)}")

# # combine and output
# fieldnames = [
#     "video_id", "channel_id", "channel_name", "lean",
#     "published_at", "year", "month",
#     "view_count", "like_count", "comment_count",
# ]

# rows = []
# for entry in all_entries:
#     vid   = entry["video_id"]
#     meta  = CHANNEL_META[entry["channel_key"]]
#     pub   = entry["published_at"]
#     stats = stats_lookup.get(vid)

#     if not stats:
#         continue  # deleted/private videos

#     rows.append({
#         "video_id":     vid,
#         "channel_id":   meta["channel_id"],
#         "channel_name": entry["channel_key"].replace("_", " ").title(),
#         "lean":         meta["lean"],
#         "published_at": pub,
#         "year":         int(pub[:4]),
#         "month":        int(pub[5:7]),
#         "view_count":   stats["view_count"],
#         "like_count":   stats["like_count"],
#         "comment_count":stats["comment_count"],
#     })
# VIDEO_OUT_FILE = "output/videos.csv"
# os.makedirs("output", exist_ok=True)
# with open(VIDEO_OUT_FILE, "w", newline="", encoding="utf-8") as f:
#     writer = csv.DictWriter(f, fieldnames=fieldnames)
#     writer.writeheader()
#     writer.writerows(rows)

# print(f"Wrote {len(rows)} rows to{VIDEO_OUT_FILE}")

Loading stats for 23338 videos
Fetching stats in 467 batches of 50
  Batch 50/467 done, 2500 stats fetched so far
  Batch 100/467 done, 5000 stats fetched so far
  Batch 150/467 done, 7500 stats fetched so far
  Batch 200/467 done, 10000 stats fetched so far
  Batch 250/467 done, 12500 stats fetched so far
  Batch 300/467 done, 15000 stats fetched so far
  Batch 350/467 done, 17500 stats fetched so far
  Batch 400/467 done, 20000 stats fetched so far
  Batch 450/467 done, 22500 stats fetched so far
  Batch 467/467 done, 23338 stats fetched so far

Stats fetched for 23338 videos.
Videos with no stats (deleted/private): 0
Wrote 23338 rows → output/videos.csv


### Create CSV file for videos (top 5 for each month) --> 8 years * 12 months * 5 videos * 2 channels = 960 videos

In [ ]:
VIDEO_OUT_FILE = "output/videos.csv"
videos_df = pd.read_csv(VIDEO_OUT_FILE)
videos_df
START = (2018, 1)  # Jan 2018 — when TYT data starts
END = (2025, 12)

videos_df = videos_df[
    (
        (videos_df["year"] > START[0])
        | ((videos_df["year"] == START[0]) & (videos_df["month"] >= START[1]))
    )
    & (
        (videos_df["year"] < END[0])
        | ((videos_df["year"] == END[0]) & (videos_df["month"] <= END[1]))
    )
]

videos_df["rank"] = videos_df.groupby(["channel_id", "year", "month"])[
    "view_count"
].rank(method="first", ascending=False)

top5 = videos_df[videos_df["rank"] <= 5].drop(columns="rank").reset_index(drop=True)


# merging video_raw with to get video titles
bs_df = pd.DataFrame(bs_data["videos"])
tyt_df = pd.DataFrame(tyt_data["videos"])
all_raw_video_df = pd.concat([bs_df, tyt_df], ignore_index=True)

top5 = top5.merge(all_raw_video_df[["video_id", "title"]], on="video_id", how="left")

display(top5.head())
display(top5.info())
top5.to_csv(
    "../../cleaned_data/youtube/top_5_videos.csv", index=False
)  # we will make a table of top 5 videos per month in oracle

top3 = videos_df[videos_df["rank"] <= 3].drop(columns="rank").reset_index(drop=True)

top3.to_csv(
    "output/top_3_videos.csv", index=False
)  # we will only take comments of the top 3 videos per month to keep the dataset manageable

,video_id,channel_id,channel_name,lean,published_at,year,month,view_count,like_count,comment_count,title
0,AC6u5Y0Ugp0,UCnQC_G5Xsjhp9fEJKuIcrSw,Ben Shapiro,conservative,2025-12-26T18:00:18Z,2025,12,688281,1193,723,Biblical Idolatry & The Role of Moses w/ Jorda...
1,r4Cig58bsr8,UCnQC_G5Xsjhp9fEJKuIcrSw,Ben Shapiro,conservative,2025-12-19T18:01:08Z,2025,12,822237,17094,12915,TPUSA Speech: Truth Over Cowardice and Grifting
2,Cfl153r3g3w,UCnQC_G5Xsjhp9fEJKuIcrSw,Ben Shapiro,conservative,2025-12-18T18:00:23Z,2025,12,637725,4134,1456,So About Trump's Speech Last Night...
3,CU6ON6aUrn0,UCnQC_G5Xsjhp9fEJKuIcrSw,Ben Shapiro,conservative,2025-12-15T18:18:00Z,2025,12,765979,8743,3606,MASSACRE: 15 Plus Killed in Islamist Hanukkah ...
4,BheW_ESM8UQ,UCnQC_G5Xsjhp9fEJKuIcrSw,Ben Shapiro,conservative,2025-12-05T18:00:15Z,2025,12,688198,6667,1222,"What Did Ilhan Omar Know, and When Did She Kno..."


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 960 entries, 0 to 959
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   video_id       960 non-null    object
 1   channel_id     960 non-null    object
 2   channel_name   960 non-null    object
 3   lean           960 non-null    object
 4   published_at   960 non-null    object
 5   year           960 non-null    int64 
 6   month          960 non-null    int64 
 7   view_count     960 non-null    int64 
 8   like_count     960 non-null    int64 
 9   comment_count  960 non-null    int64 
 10  title          960 non-null    object
dtypes: int64(5), object(6)
memory usage: 82.6+ KB


None

# Find Comments of top 3 viewed videos per month

We're gonna find the top 3 comments of the top 3 videos for each channel for each year-month. 

This results in 960 * 3 comments/video = 2880 comments hypothetically but we suspect less as some of these channels had very little following in the beginning

In [71]:
"""
If this is commented out it's because this part is done, data can be found in scripts/youtube/checkpoints/
"""

# TOP3_FILE= "output/top_3_videos.csv"
# COMMENT_OUTPUT_FILE = "../../cleaned_data/youtube/comments.csv"
# COMMENTS_PER_VIDEO = 3
# MIN_LIKES = 1

# df = pd.read_csv(TOP3_FILE)
# video_ids = df["video_id"].tolist()
# print(f"Loaded {len(video_ids)} videos from {TOP3_FILE}")

# video_ids = df["video_id"].tolist()
# print(f"Fetching comments for {len(video_ids)} videos")

# all_comments = []

# for i, video_id in enumerate(video_ids):
#     try:
#         resp = youtube.commentThreads().list(
#             part="snippet",
#             videoId=video_id,
#             order="relevance",
#             maxResults=20,
#             textFormat="plainText"
#         ).execute()

#         candidates = []
#         for item in resp.get("items", []):
#             top = item["snippet"]["topLevelComment"]["snippet"]
#             like_count = top.get("likeCount", 0)
#             if like_count >= MIN_LIKES:
#                 candidates.append({
#                     "comment_id":   item["id"],
#                     "video_id":     video_id,
#                     "text":         top.get("textDisplay", "")[:500],
#                     "like_count":   like_count,
#                     "published_at": top.get("publishedAt", ""),
#                 })

#         candidates.sort(key=lambda c: c["like_count"], reverse=True)
#         all_comments.extend(candidates[:COMMENTS_PER_VIDEO])

#     except Exception as e:
#         print(f"{video_id} failed for error: {e}")

#     if (i + 1) % 50 == 0 or (i + 1) == len(video_ids):
#         print(f"{i + 1}/{len(video_ids)} videos finished, {len(all_comments)} comments completed so far")

#     time.sleep(0.3)

# fieldnames = ["comment_id", "video_id", "text", "like_count", "published_at"]
# with open(COMMENT_OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
#     writer = csv.DictWriter(f, fieldnames=fieldnames)
#     writer.writeheader()
#     writer.writerows(all_comments)

"\nIf this is commented out it's because this part is done, data can be found in scripts/youtube/checkpoints/\n"

# all cleaned data

In [68]:
pd.read_csv("../../cleaned_data/youtube/channels.csv")

,channel_id,channel_name,lean,description,published_at,subscriber_count,total_view_count,total_video_count,uploads_playlist
0,UCnQC_G5Xsjhp9fEJKuIcrSw,Ben Shapiro,conservative,Ben Shapiro is a renowned conservative politic...,2016-11-01T00:13:29Z,7090000,4646610341,9727,UUnQC_G5Xsjhp9fEJKuIcrSw
1,UC1yBKRuGpC1tSM73A0ZjYjQ,The Young Turks,liberal,The Young Turks is The Online News Show. Join ...,2005-12-21T20:46:51Z,6500000,7307837549,71035,UU1yBKRuGpC1tSM73A0ZjYjQ


In [69]:
pd.read_csv("../../cleaned_data/youtube/top_5_videos.csv")

,video_id,channel_id,channel_name,lean,published_at,year,month,view_count,like_count,comment_count,title
0,AC6u5Y0Ugp0,UCnQC_G5Xsjhp9fEJKuIcrSw,Ben Shapiro,conservative,2025-12-26T18:00:18Z,2025,12,688281,1193,723,Biblical Idolatry & The Role of Moses w/ Jorda...
1,r4Cig58bsr8,UCnQC_G5Xsjhp9fEJKuIcrSw,Ben Shapiro,conservative,2025-12-19T18:01:08Z,2025,12,822237,17094,12915,TPUSA Speech: Truth Over Cowardice and Grifting
2,Cfl153r3g3w,UCnQC_G5Xsjhp9fEJKuIcrSw,Ben Shapiro,conservative,2025-12-18T18:00:23Z,2025,12,637725,4134,1456,So About Trump's Speech Last Night...
3,CU6ON6aUrn0,UCnQC_G5Xsjhp9fEJKuIcrSw,Ben Shapiro,conservative,2025-12-15T18:18:00Z,2025,12,765979,8743,3606,MASSACRE: 15 Plus Killed in Islamist Hanukkah ...
4,BheW_ESM8UQ,UCnQC_G5Xsjhp9fEJKuIcrSw,Ben Shapiro,conservative,2025-12-05T18:00:15Z,2025,12,688198,6667,1222,"What Did Ilhan Omar Know, and When Did She Kno..."
...,...,...,...,...,...,...,...,...,...,...,...
955,IWCzniTvdyc,UC1yBKRuGpC1tSM73A0ZjYjQ,The Young Turks,liberal,2018-01-30T03:24:00Z,2018,1,61,1,0,"The Young Turks 01.29.18: Andrew McCabe, Trump..."
956,f8KYa5n4IEw,UC1yBKRuGpC1tSM73A0ZjYjQ,The Young Turks,liberal,2018-01-20T02:41:00Z,2018,1,26,0,0,"The Young Turks 01.19.18: Opioid Crisis, Carl ..."
957,8hJJfamTeJo,UC1yBKRuGpC1tSM73A0ZjYjQ,The Young Turks,liberal,2018-01-09T02:37:00Z,2018,1,158,0,1,"The Young Turks 01.08.18: Stephen Miller, Jake..."
958,xCf9yKmV7hA,UC1yBKRuGpC1tSM73A0ZjYjQ,The Young Turks,liberal,2018-01-05T20:21:00Z,2018,1,30,1,0,"The Young Turks 01.04.18: Trump Is A Child, Ba..."


In [70]:
pd.read_csv("../../cleaned_data/youtube/comments.csv")

,comment_id,video_id,text,like_count,published_at
0,UgzphKn0fFfcKYzpuh54AaABAg,AC6u5Y0Ugp0,Wow 2 non Christians talking about Jesus and t...,253,2025-12-26T18:03:28Z
1,UgxzpFNLHkFfIylgAAh4AaABAg,AC6u5Y0Ugp0,"The Jesus Prayer,\n21:30\n""Lord Jesus Christ, ...",52,2025-12-26T19:23:54Z
2,Ugy3haJh-Wy5jXf2Uz54AaABAg,AC6u5Y0Ugp0,Why would you have someone that doesn’t believ...,41,2025-12-31T14:05:36Z
3,Ugya1VNwn6PfGwN4Pcl4AaABAg,r4Cig58bsr8,I love how Megyn Kelly went from not knowing w...,2519,2025-12-19T18:11:07Z
4,UgyhMSkBJS2O6cO5fod4AaABAg,r4Cig58bsr8,"“Peace if possible, truth at all costs.” - Mar...",1117,2025-12-21T02:19:01Z
...,...,...,...,...,...
1235,UgxgH4zvTxASFw5N4fl4AaABAg,JNFKDO4kXPg,Strange how a channel with 5.74 million subscr...,151,2024-01-29T10:32:06Z
1236,UgxQZV131jZzkyiAPXJ4AaABAg,JNFKDO4kXPg,FrazzleDrip? Let's get to that topic!!!,33,2025-01-12T17:05:50Z
1237,UgyliEX412Wj-K4Z8m14AaABAg,JNFKDO4kXPg,Frazzledrip is real. KILLARY mutilated that po...,28,2025-06-25T02:31:32Z
1238,UgxIxeU-LmiHSYJaeRN4AaABAg,pzRdnNYuLlM,Face off? Ok\nThis video is 12days old\n253 vi...,1,2024-01-23T01:06:07Z
